# AI401 — Expert Systems Lab
## Experiment 4 / Lab 5: Backward Chaining Inference Engine with Askable Facts (Goal-Driven Reasoning)
### Car-Fault Diagnosis Expert System

**Aim:**  
To implement a goal-driven backward chaining inference engine for car-fault diagnosis, query users for askable facts with an explanation facility ("WHY"), support negated conditions, and analyze the impact of hypothesis ordering.


---
### Core Implementation: Production Rule & Diagnosis Knowledge Base


In [ ]:
from typing import List, Dict, Set, Tuple, Optional, Union

class Rule:
    def __init__(self, rule_id: str, conditions: List[str], conclusion: str, description: str = ""):
        self.id = rule_id
        self.conditions = conditions
        self.conclusion = conclusion
        self.description = description or f"IF {' AND '.join(conditions)} THEN {conclusion}"

    def __repr__(self) -> str:
        return f"[{self.id}] IF {' AND '.join(self.conditions)} THEN {self.conclusion}"

def build_car_diagnosis_kb() -> Tuple[List[Rule], Dict[str, str], List[str]]:
    rules = [
        Rule("R1", ["clicking_sound", "NOT engine_cranks"], "starter_motor_fault",
             "Starter Motor Rule: Clicking sound without engine cranking indicates starter motor failure."),
        Rule("R2", ["NOT lights_working", "NOT engine_cranks"], "dead_battery",
             "Dead Battery Rule: No lights and no engine cranking indicates a dead battery."),
        Rule("R3", ["engine_cranks", "NOT engine_starts", "fuel_gauge_empty"], "empty_fuel_tank",
             "Empty Fuel Tank Rule: Engine cranks but won't start and fuel gauge is empty."),
        Rule("R4", ["engine_cranks", "NOT engine_starts", "spark_plug_dirty"], "bad_spark_plug",
             "Bad Spark Plug Rule: Engine cranks but won't start and spark plug is fouled."),
        Rule("R5", ["battery_drains_frequently", "dim_headlights"], "alternator_failure",
             "Alternator Failure Rule: Battery drains frequently and headlights are dim."),
    ]

    askable_facts = {
        "clicking_sound": "Is there a clicking sound when turning the ignition key?",
        "engine_cranks": "Does the engine crank over when turning the key?",
        "lights_working": "Are the headlights and dashboard lights working normally?",
        "engine_starts": "Does the engine start up successfully?",
        "fuel_gauge_empty": "Is the fuel gauge pointing to empty (E)?",
        "spark_plug_dirty": "Are the spark plugs dirty, oily, or fouled?",
        "battery_drains_frequently": "Does the car battery drain frequently overnight?",
        "dim_headlights": "Are the headlights dim or flickering while driving?",
    }

    hypotheses = [
        "starter_motor_fault",
        "dead_battery",
        "empty_fuel_tank",
        "bad_spark_plug",
        "alternator_failure",
    ]
    return rules, askable_facts, hypotheses


---
### Backward Chaining Engine Class with Askable Facts & WHY Facility


In [ ]:
class BackwardChainingEngine:
    def __init__(self, rules: List[Rule], askable_facts: Dict[str, str], hypotheses: List[str]):
        self.rules = rules
        self.askable_facts = askable_facts
        self.hypotheses = hypotheses
        self.wm: Dict[str, bool] = {}
        self.asked_questions: List[Tuple[str, str]] = []
        self.reasoning_stack: List[Tuple[Rule, str]] = []

    def reset_wm(self):
        self.wm.clear()
        self.asked_questions.clear()
        self.reasoning_stack.clear()

    def ask_user(self, fact: str, goal: str, simulated_inputs: Optional[List[str]] = None) -> bool:
        question = self.askable_facts.get(fact, f"Is '{fact}' true?")
        while True:
            if simulated_inputs is not None and len(simulated_inputs) > 0:
                user_input = simulated_inputs.pop(0).strip()
                print(f"[ASK] {question} -> User Input: '{user_input}'")
            else:
                user_input = input(f"[ASK] {question} (yes/no/why): ").strip()

            ans_upper = user_input.upper()
            if ans_upper == "WHY":
                print("\n" + "=" * 60)
                print("  [EXPLANATION FACILITY — 'WHY' RESPONSE]")
                print("=" * 60)
                print(f"  Target Hypothesis Being Evaluated : '{goal}'")
                if self.reasoning_stack:
                    current_rule, current_goal = self.reasoning_stack[-1]
                    print(f"  Current Rule Evaluated            : {current_rule}")
                    print(f"  Rule Description                  : {current_rule.description}")
                print(f"  Reason For Question               : To determine if condition '{fact}' is satisfied.")
                print("=" * 60 + "\n")
            elif ans_upper in ["YES", "Y", "TRUE", "1"]:
                self.wm[fact] = True
                self.asked_questions.append((fact, "yes"))
                return True
            elif ans_upper in ["NO", "N", "FALSE", "0"]:
                self.wm[fact] = False
                self.asked_questions.append((fact, "no"))
                return False
            else:
                print("  [!] Invalid response. Please answer 'yes', 'no', or 'why'.")

    def evaluate_goal(self, goal: str, simulated_inputs: Optional[List[str]] = None) -> bool:
        if goal.startswith("NOT "):
            negated_fact = goal[4:].strip()
            if negated_fact in self.wm:
                return not self.wm[negated_fact]
        elif goal in self.wm:
            return self.wm[goal]

        raw_fact = goal[4:].strip() if goal.startswith("NOT ") else goal
        is_negated = goal.startswith("NOT ")

        matching_rules = [r for r in self.rules if r.conclusion == raw_fact]
        if matching_rules:
            for rule in matching_rules:
                self.reasoning_stack.append((rule, raw_fact))
                all_conditions_satisfied = True

                for cond in rule.conditions:
                    cond_result = self.evaluate_goal(cond, simulated_inputs)
                    if not cond_result:
                        all_conditions_satisfied = False
                        break

                self.reasoning_stack.pop()

                if all_conditions_satisfied:
                    self.wm[raw_fact] = True
                    return not is_negated if is_negated else True

            if raw_fact not in self.wm:
                self.wm[raw_fact] = False
            return not is_negated if is_negated else self.wm[raw_fact]

        if raw_fact in self.askable_facts:
            answer = self.ask_user(raw_fact, goal, simulated_inputs)
            return not answer if is_negated else answer

        self.wm[raw_fact] = False
        return False

    def diagnose(self, hypotheses_order: Optional[List[str]] = None, simulated_inputs: Optional[List[str]] = None):
        self.reset_wm()
        active_hypotheses = hypotheses_order or self.hypotheses
        confirmed_fault = None
        for hypothesis in active_hypotheses:
            print(f"\n---> Investigating Hypothesis: '{hypothesis}'")
            if self.evaluate_goal(hypothesis, simulated_inputs):
                confirmed_fault = hypothesis
                print(f"  [SUCCESS] Hypothesis CONFIRMED: '{hypothesis}'")
                break
            else:
                print(f"  [REJECTED] Hypothesis failed: '{hypothesis}'")
        return confirmed_fault, self.asked_questions, self.wm


---
### Exercise 1: Starter Motor Fault Diagnosis


In [ ]:
rules, askable_facts, hypotheses = build_car_diagnosis_kb()
engine = BackwardChainingEngine(rules, askable_facts, hypotheses)

print("=" * 80)
print("EXERCISE 1: STARTER MOTOR FAULT DIAGNOSIS")
print("=" * 80)

# Simulated responses: clicking_sound -> yes, engine_cranks -> no
fault, questions, wm = engine.diagnose(simulated_inputs=["yes", "no"])

print("\nResults Summary:")
print(f"  Questions Asked   : {[q[0] + '=' + q[1] for q in questions]}")
print(f"  Working Memory    : {wm}")
print(f"  Identified Fault  : {fault}")


---
### Exercise 2: Impact of Hypothesis Order on Question Sequence


In [ ]:
print("=" * 80)
print("EXERCISE 2: IMPACT OF HYPOTHESIS ORDER ON QUESTION SEQUENCE")
print("=" * 80)

order_a = ["starter_motor_fault", "empty_fuel_tank", "dead_battery"]
print("\n>>> Testing Hypothesis Order A: " + str(order_a))
fault_a, questions_a, _ = engine.diagnose(hypotheses_order=order_a, simulated_inputs=["yes", "no"])

order_b = ["empty_fuel_tank", "starter_motor_fault", "dead_battery"]
print("\n>>> Testing Hypothesis Order B: " + str(order_b))
fault_b, questions_b, _ = engine.diagnose(hypotheses_order=order_b, simulated_inputs=["no", "yes"])

print("\nComparison Summary:")
print(f"  Order A Asked Questions : {[q[0] for q in questions_a]}")
print(f"  Order B Asked Questions : {[q[0] for q in questions_b]}")
print("  Observation: Changing hypothesis order changes the order in which questions are asked due to goal-directed reasoning.")


---
### Exercise 3: 'WHY' Explanation Facility Demonstration


In [ ]:
print("=" * 80)
print("EXERCISE 3: 'WHY' EXPLANATION FACILITY DEMONSTRATION")
print("=" * 80)

# Simulated responses: 'why' followed by 'yes', then 'no'
fault_why, questions_why, wm_why = engine.diagnose(simulated_inputs=["why", "yes", "no"])

print("\nResults Summary:")
print(f"  Final Diagnosis   : {fault_why}")
print(f"  Questions Answered: {questions_why}")
